# RLHF

RLHF improves a model with rewards while a KL leash keeps it near the pretrained behavior.

RLHF sits on the Transformer machinery from Part 8 and changes the objective around sampled responses. Attention supplies the sequence model; reward, advantage, clipping, and KL regularization decide which responses get reinforced. Save a copy to Drive to edit.

In [ ]:

import math
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 91419
rng = np.random.default_rng(SEED)
random.seed(SEED)


def sigmoid(x):
    return 1.0 / (1.0 + math.exp(-float(x)))


def softmax(values):
    arr = np.asarray(values, dtype=float)
    shifted = arr - np.max(arr)
    weights = np.exp(shifted)
    return weights / weights.sum()


def normalize_rows(matrix):
    arr = np.asarray(matrix, dtype=float)
    return arr / arr.sum(axis=1, keepdims=True)


def kl_divergence(policy, reference):
    policy = np.asarray(policy, dtype=float)
    reference = np.asarray(reference, dtype=float)
    safe_policy = np.clip(policy, 1e-9, 1.0)
    safe_reference = np.clip(reference, 1e-9, 1.0)
    return float(np.sum(safe_policy * (np.log(safe_policy) - np.log(safe_reference))))


def make_f8_ladder(topic):
    if topic == "rlhf":
        return build_rlhf_ladder()
    if topic == "dpo":
        return build_dpo_ladder()
    if topic == "constitutional":
        return build_constitutional_ladder()
    if topic == "icl":
        return build_icl_ladder()
    if topic == "prompting":
        return build_prompting_ladder()
    if topic == "reasoning":
        return build_reasoning_ladder()
    raise ValueError(topic)


def build_rlhf_ladder():
    action_names = ["concise_helpful", "verbose_loophole", "refusal"]
    return [
        {
            "name": "D1 one prompt/action",
            "actions": action_names,
            "reference": np.array([[0.62, 0.25, 0.13]]),
            "policy": np.array([[0.55, 0.32, 0.13]]),
            "reward_model": np.array([[2.0, 1.0, 0.2]]),
            "human_reward": np.array([[2.0, 1.0, 0.2]]),
            "advantages": np.array([[0.5, -0.2, -0.4]]),
            "ratios": np.array([[1.2, 0.8, 1.0]]),
            "beta": 0.10,
        },
        {
            "name": "D2 few-shot preference-policy set",
            "actions": action_names,
            "reference": normalize_rows([[0.55, 0.30, 0.15], [0.50, 0.25, 0.25]]),
            "policy": normalize_rows([[0.62, 0.28, 0.10], [0.58, 0.30, 0.12]]),
            "reward_model": np.array([[2.1, 1.4, 0.2], [1.8, 1.2, 0.5]]),
            "human_reward": np.array([[2.0, 1.0, 0.2], [1.7, 1.0, 0.7]]),
            "advantages": np.array([[0.6, -0.1, -0.5], [0.4, 0.0, -0.3]]),
            "ratios": np.array([[1.2, 0.9, 0.7], [1.1, 1.0, 0.8]]),
            "beta": 0.12,
        },
        {
            "name": "D3 distractor reward loopholes",
            "actions": action_names,
            "reference": normalize_rows([[0.48, 0.37, 0.15], [0.45, 0.35, 0.20], [0.50, 0.30, 0.20]]),
            "policy": normalize_rows([[0.44, 0.48, 0.08], [0.42, 0.45, 0.13], [0.54, 0.34, 0.12]]),
            "reward_model": np.array([[2.0, 2.6, 0.2], [1.9, 2.4, 0.3], [2.1, 1.8, 0.4]]),
            "human_reward": np.array([[2.0, 1.1, 0.2], [1.8, 1.0, 0.4], [2.0, 1.2, 0.6]]),
            "advantages": np.array([[0.4, 0.7, -0.4], [0.3, 0.6, -0.3], [0.5, 0.2, -0.2]]),
            "ratios": np.array([[1.0, 1.4, 0.7], [0.9, 1.3, 0.8], [1.1, 1.1, 0.8]]),
            "beta": 0.08,
        },
        {
            "name": "D4 real-style instruction/reward set",
            "actions": action_names,
            "reference": normalize_rows([[0.50, 0.30, 0.20], [0.45, 0.25, 0.30], [0.40, 0.35, 0.25], [0.55, 0.25, 0.20]]),
            "policy": normalize_rows([[0.63, 0.27, 0.10], [0.58, 0.27, 0.15], [0.55, 0.33, 0.12], [0.66, 0.23, 0.11]]),
            "reward_model": np.array([[2.2, 1.4, 0.5], [2.0, 1.3, 0.8], [1.9, 1.7, 0.4], [2.3, 1.2, 0.3]]),
            "human_reward": np.array([[2.1, 1.1, 0.5], [1.9, 1.0, 0.9], [1.8, 1.1, 0.5], [2.2, 1.0, 0.4]]),
            "advantages": np.array([[0.7, 0.0, -0.4], [0.5, -0.1, -0.2], [0.4, 0.1, -0.4], [0.8, -0.2, -0.5]]),
            "ratios": np.array([[1.2, 0.9, 0.7], [1.2, 1.0, 0.8], [1.1, 1.1, 0.8], [1.2, 0.8, 0.7]]),
            "beta": 0.12,
        },
        {
            "name": "D5 longer context with KL drift",
            "actions": action_names,
            "reference": normalize_rows([[0.52, 0.28, 0.20], [0.49, 0.31, 0.20], [0.46, 0.34, 0.20], [0.50, 0.27, 0.23], [0.47, 0.32, 0.21], [0.51, 0.29, 0.20]]),
            "policy": normalize_rows([[0.40, 0.55, 0.05], [0.42, 0.50, 0.08], [0.44, 0.47, 0.09], [0.61, 0.30, 0.09], [0.46, 0.45, 0.09], [0.63, 0.28, 0.09]]),
            "reward_model": np.array([[2.1, 3.0, 0.4], [2.0, 2.8, 0.4], [1.9, 2.7, 0.3], [2.2, 1.5, 0.5], [2.0, 2.5, 0.4], [2.3, 1.4, 0.3]]),
            "human_reward": np.array([[2.0, 1.0, 0.5], [1.9, 0.9, 0.5], [1.8, 1.0, 0.4], [2.1, 1.1, 0.5], [1.9, 0.8, 0.5], [2.2, 1.0, 0.4]]),
            "advantages": np.array([[0.5, 1.0, -0.3], [0.4, 0.9, -0.3], [0.3, 0.8, -0.4], [0.7, 0.1, -0.4], [0.4, 0.7, -0.3], [0.8, 0.0, -0.5]]),
            "ratios": np.array([[0.9, 1.6, 0.5], [0.9, 1.5, 0.6], [1.0, 1.4, 0.6], [1.2, 1.0, 0.6], [1.0, 1.4, 0.6], [1.2, 0.9, 0.6]]),
            "beta": 0.04,
        },
    ]


def build_dpo_ladder():
    return [
        {
            "name": "D1 one prompt preference",
            "policy_chosen": np.array([-1.0]),
            "policy_rejected": np.array([-2.0]),
            "reference_chosen": np.array([-1.5]),
            "reference_rejected": np.array([-2.2]),
            "labels": np.array([1]),
            "beta": 2.0,
        },
        {
            "name": "D2 few-shot preference set",
            "policy_chosen": np.array([-0.8, -1.1, -1.0]),
            "policy_rejected": np.array([-1.8, -1.7, -2.0]),
            "reference_chosen": np.array([-1.2, -1.4, -1.5]),
            "reference_rejected": np.array([-1.9, -1.9, -2.2]),
            "labels": np.array([1, 1, 1]),
            "beta": 2.0,
        },
        {
            "name": "D3 label noise and distractors",
            "policy_chosen": np.array([-0.7, -1.8, -0.9, -1.3]),
            "policy_rejected": np.array([-1.6, -1.4, -1.7, -1.2]),
            "reference_chosen": np.array([-1.1, -1.5, -1.4, -1.4]),
            "reference_rejected": np.array([-1.8, -1.8, -2.0, -1.6]),
            "labels": np.array([1, 0, 1, 0]),
            "beta": 2.0,
        },
        {
            "name": "D4 real-style pair corpus",
            "policy_chosen": np.array([-0.7, -0.9, -1.1, -0.8, -1.0]),
            "policy_rejected": np.array([-1.8, -1.5, -1.6, -1.7, -1.9]),
            "reference_chosen": np.array([-1.2, -1.2, -1.4, -1.3, -1.5]),
            "reference_rejected": np.array([-1.9, -1.7, -1.9, -1.8, -2.1]),
            "labels": np.array([1, 1, 1, 1, 1]),
            "beta": 1.5,
        },
        {
            "name": "D5 longer context with saturated beta",
            "policy_chosen": np.array([-0.5, -0.6, -2.2, -0.7, -2.0, -0.9]),
            "policy_rejected": np.array([-2.0, -1.9, -1.1, -1.8, -1.0, -1.6]),
            "reference_chosen": np.array([-1.2, -1.3, -1.8, -1.4, -1.7, -1.5]),
            "reference_rejected": np.array([-2.1, -2.0, -1.7, -2.0, -1.6, -1.9]),
            "labels": np.array([1, 1, 0, 1, 0, 1]),
            "beta": 6.0,
        },
    ]


def build_constitutional_ladder():
    principles = ["harmlessness", "honesty", "helpfulness"]
    return [
        {
            "name": "D1 one prompt/principle",
            "principles": principles[:2],
            "weights": np.array([2.0, 1.0]),
            "violations": np.array([[0.3, 0.8]]),
            "revised": np.array([[0.2, 0.2]]),
            "task_success": np.array([0.90]),
        },
        {
            "name": "D2 few-shot critique set",
            "principles": principles,
            "weights": np.array([2.0, 1.0, 1.5]),
            "violations": np.array([[0.4, 0.5, 0.2], [0.2, 0.8, 0.3]]),
            "revised": np.array([[0.2, 0.2, 0.2], [0.2, 0.3, 0.2]]),
            "task_success": np.array([0.88, 0.86]),
        },
        {
            "name": "D3 conflicting principles and distractors",
            "principles": principles,
            "weights": np.array([2.5, 1.0, 0.8]),
            "violations": np.array([[0.6, 0.3, 0.7], [0.5, 0.6, 0.6], [0.2, 0.4, 0.8]]),
            "revised": np.array([[0.2, 0.2, 0.5], [0.2, 0.3, 0.5], [0.1, 0.2, 0.6]]),
            "task_success": np.array([0.70, 0.72, 0.68]),
        },
        {
            "name": "D4 real-style policy examples",
            "principles": principles,
            "weights": np.array([2.0, 1.3, 1.4]),
            "violations": np.array([[0.5, 0.2, 0.3], [0.2, 0.7, 0.4], [0.4, 0.5, 0.5], [0.3, 0.2, 0.8]]),
            "revised": np.array([[0.2, 0.2, 0.3], [0.2, 0.2, 0.3], [0.2, 0.2, 0.4], [0.2, 0.2, 0.5]]),
            "task_success": np.array([0.86, 0.88, 0.84, 0.82]),
        },
        {
            "name": "D5 longer context with vague scores",
            "principles": principles,
            "weights": np.array([3.0, 1.0, 0.3]),
            "violations": np.array([[0.7, 0.3, 0.8], [0.6, 0.4, 0.9], [0.5, 0.5, 0.8], [0.4, 0.6, 0.9], [0.7, 0.2, 0.7]]),
            "revised": np.array([[0.1, 0.2, 0.7], [0.1, 0.3, 0.8], [0.1, 0.3, 0.7], [0.1, 0.4, 0.8], [0.1, 0.2, 0.6]]),
            "task_success": np.array([0.46, 0.42, 0.48, 0.40, 0.44]),
        },
    ]


def build_icl_ladder():
    return [
        {
            "name": "D1 one prompt",
            "scores": np.array([[2.0, 1.0, 0.0]]),
            "labels": np.array([[1.0, 0.0, 1.0]]),
            "targets": np.array([1]),
            "token_budget": 8,
            "demo_tokens": 6,
            "recency_bonus": 0.5,
        },
        {
            "name": "D2 few-shot set",
            "scores": np.array([[2.2, 1.0, 0.1], [0.4, 1.8, 0.2], [1.7, 0.8, 0.5]]),
            "labels": np.array([[1.0, 0.0, 1.0], [0.0, 1.0, 0.0], [1.0, 0.0, 0.0]]),
            "targets": np.array([1, 1, 1]),
            "token_budget": 16,
            "demo_tokens": 8,
            "recency_bonus": 0.4,
        },
        {
            "name": "D3 distractors and order flips",
            "scores": np.array([[2.0, 1.9, 0.1, 0.0], [0.2, 1.6, 1.5, 0.1], [1.4, 0.3, 1.3, 0.2], [0.1, 0.2, 1.7, 1.6]]),
            "labels": np.array([[1.0, 0.0, 0.0, 1.0], [0.0, 1.0, 0.0, 0.0], [1.0, 0.0, 0.0, 1.0], [0.0, 1.0, 0.0, 0.0]]),
            "targets": np.array([1, 1, 1, 1]),
            "token_budget": 24,
            "demo_tokens": 16,
            "recency_bonus": 0.6,
        },
        {
            "name": "D4 real text-label examples",
            "scores": np.array([[2.3, 1.2, 0.6, 0.1], [0.5, 2.1, 1.0, 0.2], [1.9, 0.8, 0.7, 0.4], [0.4, 1.7, 1.4, 0.3], [2.1, 0.7, 0.8, 0.2]]),
            "labels": np.array([[1.0, 0.0, 1.0, 0.0], [0.0, 1.0, 1.0, 0.0], [1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0], [1.0, 0.0, 1.0, 0.0]]),
            "targets": np.array([1, 1, 1, 1, 1]),
            "token_budget": 40,
            "demo_tokens": 24,
            "recency_bonus": 0.3,
        },
        {
            "name": "D5 longer context with diluted attention",
            "scores": np.array([[2.0, 1.9, 1.8, 1.7, 0.2, 0.1], [0.3, 1.8, 1.7, 1.6, 0.2, 0.1], [1.7, 1.6, 1.5, 1.4, 0.3, 0.2], [0.2, 1.7, 1.6, 1.5, 0.3, 0.2], [1.8, 1.7, 1.6, 1.5, 0.3, 0.2], [0.2, 1.6, 1.5, 1.4, 0.3, 0.2]]),
            "labels": np.array([[1.0, 0.0, 0.0, 0.0, 1.0, 1.0], [0.0, 1.0, 0.0, 0.0, 1.0, 0.0], [1.0, 0.0, 0.0, 0.0, 1.0, 0.0], [0.0, 1.0, 0.0, 0.0, 1.0, 0.0], [1.0, 0.0, 0.0, 0.0, 1.0, 0.0], [0.0, 1.0, 0.0, 0.0, 1.0, 0.0]]),
            "targets": np.array([1, 1, 1, 1, 1, 1]),
            "token_budget": 64,
            "demo_tokens": 54,
            "recency_bonus": 0.7,
        },
    ]


def build_prompting_ladder():
    return [
        {
            "name": "D1 one prompt",
            "base_logits": np.array([[1.0, 0.0]]),
            "demo_boost": 0.8,
            "targets": np.array([0]),
            "prompt_tokens": 1200,
            "window": 2048,
            "format_penalty": -1.0,
            "cot_success": 0.6,
            "samples": 3,
            "imbalance": 0.0,
        },
        {
            "name": "D2 few-shot set",
            "base_logits": np.array([[0.9, 0.0], [0.2, 0.7], [0.8, 0.1]]),
            "demo_boost": 0.7,
            "targets": np.array([0, 1, 0]),
            "prompt_tokens": 900,
            "window": 2048,
            "format_penalty": -0.5,
            "cot_success": 0.62,
            "samples": 3,
            "imbalance": 0.1,
        },
        {
            "name": "D3 distractors and label imbalance",
            "base_logits": np.array([[0.4, 0.2], [0.1, 0.3], [0.5, 0.3], [0.2, 0.4]]),
            "demo_boost": 0.5,
            "targets": np.array([0, 1, 0, 1]),
            "prompt_tokens": 1500,
            "window": 2048,
            "format_penalty": -1.0,
            "cot_success": 0.55,
            "samples": 5,
            "imbalance": 0.6,
        },
        {
            "name": "D4 real-style QA/instruction corpus",
            "base_logits": np.array([[0.9, 0.1], [0.2, 0.8], [0.7, 0.2], [0.1, 0.7], [0.8, 0.0]]),
            "demo_boost": 0.6,
            "targets": np.array([0, 1, 0, 1, 0]),
            "prompt_tokens": 1300,
            "window": 4096,
            "format_penalty": -0.4,
            "cot_success": 0.66,
            "samples": 5,
            "imbalance": 0.0,
        },
        {
            "name": "D5 longer context with ungrounded chains",
            "base_logits": np.array([[0.5, 0.4], [0.4, 0.5], [0.6, 0.5], [0.3, 0.4], [0.5, 0.5], [0.4, 0.6]]),
            "demo_boost": 0.3,
            "targets": np.array([0, 1, 0, 1, 0, 1]),
            "prompt_tokens": 3800,
            "window": 4096,
            "format_penalty": -1.2,
            "cot_success": 0.52,
            "samples": 7,
            "imbalance": 0.8,
        },
    ]


def build_reasoning_ladder():
    return [
        {
            "name": "D1 one prompt",
            "votes": ["A", "A", "B", "A", "B"],
            "correct": "A",
            "branching": 3,
            "depth": 2,
            "checks": [("path1", "claim_shared"), ("path1", "claim_a"), ("path2", "claim_shared"), ("path2", "claim_b")],
            "prior": 0.55,
            "likelihood_ratio": 3.0,
            "tokens_per_trace": 100,
        },
        {
            "name": "D2 few-shot reasoning set",
            "votes": ["A", "C", "A", "A", "B", "A"],
            "correct": "A",
            "branching": 2,
            "depth": 3,
            "checks": [("p1", "sum"), ("p2", "sum"), ("p3", "unit"), ("p4", "lookup")],
            "prior": 0.60,
            "likelihood_ratio": 2.0,
            "tokens_per_trace": 80,
        },
        {
            "name": "D3 distractor correlated errors",
            "votes": ["B", "B", "A", "B", "A", "B"],
            "correct": "A",
            "branching": 3,
            "depth": 2,
            "checks": [("p1", "bad_hint"), ("p2", "bad_hint"), ("p3", "arithmetic"), ("p4", "bad_hint"), ("p5", "arithmetic")],
            "prior": 0.45,
            "likelihood_ratio": 1.5,
            "tokens_per_trace": 90,
        },
        {
            "name": "D4 real-style arithmetic/tool set",
            "votes": ["A", "A", "A", "C", "A", "B", "A"],
            "correct": "A",
            "branching": 3,
            "depth": 3,
            "checks": [("p1", "calc"), ("p2", "calc"), ("p3", "unit"), ("p4", "calendar"), ("p5", "unit"), ("p6", "lookup")],
            "prior": 0.62,
            "likelihood_ratio": 2.8,
            "tokens_per_trace": 110,
        },
        {
            "name": "D5 longer context with weak scorer",
            "votes": ["B", "B", "B", "A", "A", "B", "C", "B"],
            "correct": "A",
            "branching": 4,
            "depth": 3,
            "checks": [("p1", "misread"), ("p2", "misread"), ("p3", "misread"), ("p4", "calc"), ("p5", "lookup"), ("p6", "misread"), ("p7", "calc"), ("p8", "lookup")],
            "prior": 0.50,
            "likelihood_ratio": 1.2,
            "tokens_per_trace": 140,
        },
    ]


def ladder_frame(ladder):
    rows = []
    for rung in ladder:
        keys = [key for key in rung.keys() if key != "name"]
        size = 0
        for value in rung.values():
            if isinstance(value, np.ndarray):
                size = max(size, int(value.size))
        rows.append({"rung": rung["name"], "size": size, "fields": ", ".join(keys[:5])})
    return pd.DataFrame(rows)


We first build the reusable RLHF accounting function. It computes the reward-minus-KL objective, the PPO surrogate, and a human win check from tiny policy tables rather than from any large model.

In [ ]:

def kl_regularized_update(reward, kl_value, beta, ratio, advantage, clip_range=0.2):
    objective = reward - beta * kl_value
    clipped_ratio = float(np.clip(ratio, 1.0 - clip_range, 1.0 + clip_range))
    raw_ppo = ratio * advantage
    clipped_ppo = clipped_ratio * advantage
    ppo_term = min(raw_ppo, clipped_ppo)
    baseline = reward - advantage
    return {
        "objective": objective,
        "clipped_ratio": clipped_ratio,
        "ppo_term": ppo_term,
        "baseline": baseline,
    }


The lesson objective is $J(\theta)=\mathbb{E}[r(x,y)]-\beta\,\mathrm{KL}(\pi_\theta(\cdot\mid x)\Vert\pi_0(\cdot\mid x))$. We also use PPO's clipped ratio term so a sampled response cannot dominate the update.

Plug in the plan's numbers: reward $2.0$ minus KL price $0.1\times3.0$ gives $1.700$; ratio $1.2$ with advantage $0.5$ gives $0.600$; ratio $1.4$ clips to $1.2$; doubled KL raises penalty $0.3$ to $0.6$; advantage $2.0-1.2=0.800$.

In [ ]:

check = kl_regularized_update(2.0, 3.0, 0.1, 1.2, 0.5)
clipped = kl_regularized_update(2.0, 3.0, 0.1, 1.4, 0.5)
penalty_small = 0.1 * 3.0
penalty_large = 0.1 * 6.0
advantage = 2.0 - 1.2
assert round(check["objective"], 3) == 1.700
assert round(check["ppo_term"], 3) == 0.600
assert round(clipped["clipped_ratio"], 3) == 1.200
assert round(clipped["ppo_term"], 3) == 0.600
assert round(penalty_small, 3) == 0.300
assert round(penalty_large, 3) == 0.600
assert round(advantage, 3) == 0.800
print(check)


## The dataset ladder
Build the F8 D1-D5 ladder inline. Each rung increases context, distractors, policy pressure, or reasoning complexity while staying tiny and CPU-only.

In [ ]:

ladder = make_f8_ladder("rlhf")
preview = ladder_frame(ladder)
print(preview.to_string(index=False))
print("sample rung")
print(ladder[0])


## Run the same method across D1-D5
The same method is applied to every rung; only the rung data changes.

In [ ]:

def evaluate_rlhf_rung(rung, beta=None):
    beta_value = rung["beta"] if beta is None else beta
    policy = rung["policy"]
    reference = rung["reference"]
    reward_model = rung["reward_model"]
    human_reward = rung["human_reward"]
    expected_reward = np.sum(policy * reward_model, axis=1)
    human_expected = np.sum(policy * human_reward, axis=1)
    kl_values = np.array([kl_divergence(policy[i], reference[i]) for i in range(policy.shape[0])])
    objective = expected_reward - beta_value * kl_values
    chosen = np.argmax(policy, axis=1)
    human_best = np.argmax(human_reward, axis=1)
    win_rate = np.mean(chosen == human_best)
    return {
        "objective": float(np.mean(objective)),
        "reward": float(np.mean(expected_reward)),
        "human_reward": float(np.mean(human_expected)),
        "kl": float(np.mean(kl_values)),
        "win_rate": float(win_rate),
    }


results = []
for rung in ladder:
    row = evaluate_rlhf_rung(rung)
    row["rung"] = rung["name"]
    results.append(row)

results_df = pd.DataFrame(results)
print(results_df[["rung", "objective", "reward", "kl", "win_rate"]].to_string(index=False))


## Results visualization
First inspect per-rung artifacts, then a summary curve for the plan metric.

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for index, rung in enumerate(ladder):
    policy_mean = rung["policy"].mean(axis=0)
    reward_mean = rung["reward_model"].mean(axis=0)
    axes[0, index].bar(rung["actions"], policy_mean)
    axes[0, index].set_title(rung["name"].split()[0])
    axes[0, index].tick_params(axis="x", rotation=45)
    axes[0, index].set_ylim(0, 0.7)
    axes[1, index].bar(rung["actions"], reward_mean, color="tab:orange")
    axes[1, index].tick_params(axis="x", rotation=45)
    axes[1, index].set_ylim(0, 3.2)
fig.suptitle("Per-rung policy mass and reward-model scores")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, 6), results_df["win_rate"], marker="o", label="human win rate")
ax.plot(range(1, 6), results_df["objective"], marker="s", label="reward minus KL")
ax.set_xticks(range(1, 6))
ax.set_xlabel("D-rung")
ax.set_title("Win-rate and KL-regularized reward across D1-D5")
ax.legend()
plt.show()


## Pitfall on the hardest rung
Pitfall on D5: if $\beta$ is too low, the policy chases the reward-model loophole action. The fix is to raise the KL leash and verify against held-out human reward, not just the reward model.

In [ ]:

d5 = ladder[-1]
loose = evaluate_rlhf_rung(d5, beta=0.01)
tight = evaluate_rlhf_rung(d5, beta=0.30)
loophole_share = float(d5["policy"][:, 1].mean())
fixed_policy = d5.copy()
fixed_policy["policy"] = normalize_rows(0.65 * d5["policy"] + 0.35 * d5["reference"])
fixed_policy["beta"] = 0.30
fixed = evaluate_rlhf_rung(fixed_policy)
print("loose beta objective", round(loose["objective"], 3))
print("loophole action share", round(loophole_share, 3))
print("tight beta objective", round(tight["objective"], 3))
print("fixed human reward", round(fixed["human_reward"], 3))
print("fixed KL", round(fixed["kl"], 3))



## Evaluate it + Practice
- Metric: reward minus KL and human win rate; compare against a no-skill baseline such as always choosing the reference/default answer.
- Sanity check: D1 must reproduce the exact lesson arithmetic asserted above before you trust D5.
- Ablation: turn off the key idea (KL anchor, reference margin, helpfulness term, retrieved examples, balanced prompt, or diversified traces) and confirm the metric drops or the failure mode appears.
- Failure signals: saturated probabilities, zero token budget, high reward with low human win rate, or improved safety with collapsed helpfulness.
- Keep everything CPU-only and seeded; do not download models or execute training-heavy notebook code.


Practice: Change beta on D5 and find the smallest value that reduces loophole share.

Practice: Replace the reward model with the held-out human rewards and compare action choices.

Practice: Plot PPO clipped versus unclipped terms for every D5 action.